# Twitter Sentiment Analysis

In [1]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import re
import pickle

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

nltk.download('wordnet')
nltk.download('stopwords')

DATA_PATH = 'data/dataset.csv'
ARTIFACT_DIR = 'artifacts'
os.makedirs(ARTIFACT_DIR, exist_ok=True)

COLUMNS = ['Sentiment', 'Id', 'Date', 'Flag', 'User', 'Tweet']


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nikun\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nikun\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
# Load raw dataset

dataset = pd.read_csv(DATA_PATH, names=COLUMNS, encoding='latin-1')
print('Shape:', dataset.shape)
print(dataset[['Sentiment', 'Tweet']].head())


Shape: (1600000, 6)
   Sentiment                                              Tweet
0          0  @switchfoot http://twitpic.com/2y1zl - Awww, t...
1          0  is upset that he can't update his Facebook by ...
2          0  @Kenichan I dived many times for the ball. Man...
3          0    my whole body feels itchy and like its on fire 
4          0  @nationwideclass no, it's not behaving at all....


In [4]:
# Train / test split

X_train, X_test, y_train, y_test = train_test_split(
    dataset['Tweet'],
    dataset['Sentiment'],
    test_size=0.20,
    random_state=100,
)

train_df = pd.DataFrame({'Tweet': X_train, 'Sentiment': y_train})
test_df = pd.DataFrame({'Tweet': X_test, 'Sentiment': y_test})

print('Train size:', train_df.shape)
print('Test size:', test_df.shape)


Train size: (1280000, 2)
Test size: (320000, 2)


In [5]:
# Text preprocessing helpers

STOPWORDS = set(stopwords.words('english'))
if 'not' in STOPWORDS:
    STOPWORDS.remove('not')

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()


def expand_tweet(tokens):
    expanded = []
    for word in tokens:
        if re.search("n't", word):
            expanded.append(word.split("n't")[0])
            expanded.append('not')
        else:
            expanded.append(word)
    return expanded


def clean_tweet_series(series):
    s = series.astype(str)
    s = s.str.replace(r"@[\w]*", "", regex=True)
    s = s.str.replace(r"[^a-zA-Z' ]", "", regex=True)
    s = s.replace(re.compile(r"((www\.[^\s]+)|(https?://[^\s]+))"), "")
    s = s.replace(re.compile(r"(^| ).( |$)"), " ")
    tokens = s.str.split()
    tokens = tokens.apply(lambda tweet: [w for w in tweet if w not in STOPWORDS])
    tokens = tokens.apply(expand_tweet)
    tokens = tokens.apply(lambda tweet: [lemmatizer.lemmatize(w) for w in tweet])
    tokens = tokens.apply(lambda tweet: [stemmer.stem(w) for w in tweet])
    return tokens.apply(lambda tweet: ' '.join(tweet))


In [6]:
# Apply preprocessing

train_df = train_df.copy()
test_df = test_df.copy()

train_df['Clean_tweet'] = clean_tweet_series(train_df['Tweet'])
test_df['Clean_tweet'] = clean_tweet_series(test_df['Tweet'])

print(train_df[['Tweet', 'Clean_tweet']].head())


                                                     Tweet  \
338461   @SweetCandiesXXX if u came to visit here in 17...   
986385   thanks @Just4Julia! good advice for this day  ...   
1301957  @RogJ  Thank you, Roger! Oh, and very nice to ...   
1003165  @MattMazur Hi Matt, how are you today? I am im...   
1369722             @MrsNickJonass that's cool, i like it    

                                               Clean_tweet  
338461                                    came visit choic  
986385   thank good advic day quotsmil fear sorrow smil...  
1301957                            thank roger oh nice see  
1003165                  hi matt today improv french tweet  
1369722                                    that' cool like  


In [7]:
# Tokenization and padding

MAX_WORDS = 2000

tokenizer = Tokenizer(num_words=MAX_WORDS, split=' ')
tokenizer.fit_on_texts(train_df['Clean_tweet'].astype(str).values)

train_sequences = tokenizer.texts_to_sequences(train_df['Clean_tweet'].astype(str).values)
max_len = max(len(seq) for seq in train_sequences)

X_train_seq = pad_sequences(train_sequences, maxlen=max_len)
X_test_seq = pad_sequences(
    tokenizer.texts_to_sequences(test_df['Clean_tweet'].astype(str).values),
    maxlen=max_len,
)

y_train_cat = pd.get_dummies(train_df['Sentiment']).values
y_test_cat = pd.get_dummies(test_df['Sentiment']).values

print('Vocab size:', len(tokenizer.word_index))
print('Max sequence length:', max_len)


Vocab size: 373700
Max sequence length: 40


In [ ]:
# Build and train LSTM model

model = Sequential()
model.add(Embedding(MAX_WORDS, 128, input_length=X_train_seq.shape[1]))
model.add(SpatialDropout1D(0.4))
model.add(LSTM(256, dropout=0.2))
model.add(Dense(2, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

history = model.fit(
    X_train_seq,
    y_train_cat,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
)

score, accuracy = model.evaluate(X_test_seq, y_test_cat, batch_size=128)
print('Test accuracy:', accuracy)


C:\Users\nikun\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
8000/8000 ━━━━━━━━━━━━━━━━━━━━ 1795s 224ms/step - accuracy: 0.7672 - loss: 0.4808 - val_accuracy: 0.7775 - val_loss: 0.4629
Epoch 2/10
8000/8000 ━━━━━━━━━━━━━━━━━━━━ 3465s 433ms/step - accuracy: 0.7789 - loss: 0.4621 - val_accuracy: 0.7813 - val_loss: 0.4576
Epoch 3/10
 608/8000 ━━━━━━━━━━━━━━━━━━━━ 50:25 409ms/step - accuracy: 0.7849 - loss: 0.4528

In [7]:
# Save model and tokenizer

model_path = os.path.join(ARTIFACT_DIR, 'sentiment_lstm.h5')
tokenizer_path = os.path.join(ARTIFACT_DIR, 'tokenizer.pkl')
meta_path = os.path.join(ARTIFACT_DIR, 'meta.npy')

model.save(model_path)
with open(tokenizer_path, 'wb') as f:
    pickle.dump(tokenizer, f)

np.save(meta_path, {'max_len': X_train_seq.shape[1]})

print('Saved model to', model_path)
print('Saved tokenizer to', tokenizer_path)


NameError: name 'model' is not defined

In [10]:
# Load model and run sample predictions
from tensorflow.keras.models import  load_model
model_path = os.path.join(ARTIFACT_DIR, 'sentiment_lstm.h5')
loaded_model = load_model(model_path)
with open(tokenizer_path, 'rb') as f:
    loaded_tokenizer = pickle.load(f)

meta = np.load(meta_path, allow_pickle=True).item()
max_len_loaded = meta['max_len']


def predict_sentiment(text):
    seq = loaded_tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len_loaded)
    probs = loaded_model.predict(padded)[0]
    label = int(np.argmax(probs))
    return label, float(probs[label])


sample_text = "this is worst product ever"
label, confidence = predict_sentiment(sample_text)
print('Text:', sample_text)
print('Predicted sentiment:', label)
print('Confidence:', confidence)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 277ms/step
Text: this is worst product ever
Predicted sentiment: 0
Confidence: 0.9682789444923401
